<a href="https://colab.research.google.com/github/kulkarnipranjal55-code/AI-Powered-HVAC-Decision-Intelligence-Engine/blob/main/AI_Powered_HVAC_Decision_Intelligence_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# HVAC DECISION INTELLIGENCE — FINAL HACKATHON PIPELINE
#importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

from sklearn.ensemble import GradientBoostingRegressor, IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, classification_report

np.random.seed(42)

DATA_PATH = "/content/"

month_map = {
"Jan":1,"Feb":2,"Mar":3,"Apr":4,"May":5,"Jun":6,
"Jul":7,"Aug":8,"Sep":9,"Oct":10,"Nov":11,"Dec":12
}


#Data splitting function:
def split_data(df):
    train = df[(df["year"]>=2022) & (df["year"]<=2023)].copy()
    val   = df[(df["year"]==2024)].copy()
    test  = df[(df["year"]==2025)].copy()
    return train,val,test


# 1️)DEMAND FORECASTING WITH EXTERNAL SIGNALS

def demand_forecasting():

    print("\n===== DEMAND FORECASTING =====")

    demand = pd.read_csv(DATA_PATH+"demand_history.csv")
    external = pd.read_csv(DATA_PATH+"external_signals.csv")

    demand_long = demand.melt(
        id_vars=["product","aps","year"],
        var_name="month",
        value_name="sales"
    )

    demand_long["month"]=demand_long["month"].map(month_map)

    data = demand_long.merge(external,on=["year","month"],how="left")
    data = data.sort_values(["product","year","month"])

    # Feature Engineering
    data["lag1"]=data.groupby("product")["sales"].shift(1)
    data["lag12"]=data.groupby("product")["sales"].shift(12)
    data["roll3"]=data.groupby("product")["sales"].shift(1).rolling(3).mean()

    data["month_sin"]=np.sin(2*np.pi*data["month"]/12)
    data["month_cos"]=np.cos(2*np.pi*data["month"]/12)

    data=data.dropna()

    train,val,test = split_data(data)

    features=[
        "lag1","lag12","roll3",
        "avg_temp_f","cooling_degree_days",
        "housing_starts_k","gdp_growth_pct",
        "month_sin","month_cos"
    ]

    model=GradientBoostingRegressor(
        n_estimators=220,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    model.fit(train[features],train["sales"])

    val_preds=model.predict(val[features])
    test_preds=model.predict(test[features])

    print("VAL MAPE:",mean_absolute_percentage_error(val["sales"],val_preds))
    print("TEST MAPE:",mean_absolute_percentage_error(test["sales"],test_preds))

    # Visualization
    test["prediction"]=test_preds
    test["date"]=pd.to_datetime(test["year"].astype(str)+"-"+test["month"].astype(str)+"-01")

    plt.figure(figsize=(12,5))
    plt.plot(test["date"],test["sales"],label="Actual")
    plt.plot(test["date"],test["prediction"],label="Forecast",linestyle="--")
    plt.legend()
    plt.title("HVAC Demand Forecast")
    plt.show()

    # SHAP
    explainer=shap.Explainer(model)
    shap_values=explainer(test[features])
    shap.summary_plot(shap_values,test[features])



# 2️)ANOMALY DETECTION IN SALES PATTERNS

def anomaly_detection():

    print("\n===== ANOMALY DETECTION =====")

    demand=pd.read_csv(DATA_PATH+"demand_history.csv")
    external=pd.read_csv(DATA_PATH+"external_signals.csv")
    promo=pd.read_csv(DATA_PATH+"promotion_history.csv")
    capacity=pd.read_csv(DATA_PATH+"capacity_constraints.csv")
    labels=pd.read_csv(DATA_PATH+"anomaly_labels.csv")

    demand_long=demand.melt(
        id_vars=["product","aps","year"],
        var_name="month",
        value_name="sales"
    )

    demand_long["month"]=demand_long["month"].map(month_map)
    promo["month"]=promo["month"].map(month_map)

    data=demand_long.merge(external,on=["year","month"],how="left")
    data=data.merge(promo,on=["product","year","month"],how="left")
    data=data.merge(capacity,on=["product","year","month"],how="left")

    data.fillna(0,inplace=True)

    # Features
    data["lag1"]=data.groupby("product")["sales"].shift(1)
    data["lag12"]=data.groupby("product")["sales"].shift(12)
    data["roll3"]=data.groupby("product")["sales"].shift(1).rolling(3).mean()
    data["zscore"]=data.groupby("product")["sales"].transform(lambda x:(x-x.mean())/(x.std()+1e-6))

    data=data.dropna()

    train,val,test = split_data(data)

    features=["sales","lag1","lag12","roll3","zscore","avg_temp_f","gdp_growth_pct"]

    scaler=StandardScaler()
    scaler.fit(train[features])

    X_train=scaler.transform(train[features])
    X_val=scaler.transform(val[features])
    X_test=scaler.transform(test[features])

    model=IsolationForest(contamination=0.05,n_estimators=200,random_state=42)

    train["is_anomaly"]=(model.fit_predict(X_train)==-1).astype(int)
    val["is_anomaly"]=(model.predict(X_val)==-1).astype(int)
    test["is_anomaly"]=(model.predict(X_test)==-1).astype(int)

    full=pd.concat([train,val,test])

    # Explanation Engine

    def explain(row):

        reasons=[]

        if abs(row["zscore"])>2:
            reasons.append("Demand Spike/Drop")

        if row.get("promo_flag",0)==1:
            reasons.append("Promotion")

        if row.get("capacity_limited",0)==1:
            reasons.append("Capacity Issue")

        if row["avg_temp_f"]>85:
            reasons.append("Extreme Weather")

        return ", ".join(reasons) if reasons else "Pattern Shift"

    full["explanation"]=full.apply(explain,axis=1)


    # Time Series Plot
    # Creating proper date column
    full["date"] = pd.to_datetime(
    full["year"].astype(str) + "-" +
    full["month"].astype(str) + "-01"
    )
    # Sorting properly
    full = full.sort_values(["product","date"])

    # Ploting ONE product clearly
    product_to_plot = full["product"].unique()[0]
    df_p = full[full["product"] == product_to_plot]

    plt.figure(figsize=(14,6))
    plt.plot(df_p["date"], df_p["sales"], label="Sales")

    anoms = df_p[df_p["is_anomaly"]==1]
    plt.scatter(anoms["date"], anoms["sales"], label="Anomaly")

    plt.title(f"Anomaly Timeline - {product_to_plot}")
    plt.xlabel("Date")
    plt.ylabel("Sales")
    plt.legend()
    plt.show()

    # PIE CHART of % ANOMALIES with reason behind it
    # Count normal cases
    normal_count = (full["is_anomaly"]==0).sum()

    # Count anomaly reasons
    reason_counts = (
    full[full["is_anomaly"]==1]["explanation"]
    .str.split(", ")
    .explode()
    .value_counts()
    )

    # Combine into ONE series
    pie_labels = ["Normal"] + ["Anomaly — " + r for r in reason_counts.index]
    pie_values = [normal_count] + reason_counts.values.tolist()

    plt.figure(figsize=(7,7))
    plt.pie(
        pie_values,
        labels=pie_labels,
        autopct="%1.0f%%"
        )

    plt.title("Anomaly Percentage & Reasons Breakdown")
    plt.show()

    # 3️)MARKET SHARE INTELLIGENCE

    def market_share_intelligence():

      print("\n===== MARKET SHARE INTELLIGENCE =====")
      market=pd.read_csv(DATA_PATH+"market_share_history.csv")
      external=pd.read_csv(DATA_PATH+"external_signals.csv")
      news=pd.read_json(DATA_PATH+"news_corpus.json")

      news["text"]=news["text"].astype(str).str.lower()
      news["launch_signal"]=news["text"].str.contains("launch|release").astype(int)
      news["pricing_signal"]=news["text"].str.contains("price|discount").astype(int)
      news["supply_signal"]=news["text"].str.contains("delay|shortage").astype(int)
      news["regulation_signal"]=news["text"].str.contains("regulation|standard").astype(int)
      signals=news.groupby(["year","month"])[
          ["launch_signal","pricing_signal","supply_signal","regulation_signal"]
          ].sum().reset_index()
          market_long=market.melt(
              id_vars=["product","year"],
              var_name="month",
              value_name="market_share"
              )

          market_long["month"]=market_long["month"].str.capitalize().map(month_map)
          data=market_long.merge(external,on=["year","month"],how="left")
          data=data.merge(signals,on=["year","month"],how="left")

          data.fillna(0,inplace=True)

          data["lag1"]=data.groupby("product")["market_share"].shift(1)

          data["lag3"]=data.groupby("product")["market_share"].shift(3)

          data["month_sin"]=np.sin(2*np.pi*data["month"]/12)

          data["month_cos"]=np.cos(2*np.pi*data["month"]/12)

          data=data.dropna()

          data["target_share"]=data.groupby("product")["market_share"].shift(-3)
          data=data.dropna()

          train,val,test = split_data(data)

          features=[
              "lag1","lag3",
              "avg_temp_f","gdp_growth_pct",
              "launch_signal","pricing_signal","supply_signal","regulation_signal",
              "month_sin","month_cos"
              ]
              model=GradientBoostingRegressor(
                  n_estimators=200,
                  learning_rate=0.04,
                  max_depth=3,
                  random_state=42
                  )
              model.fit(train[features],train["target_share"])
              val_preds=model.predict(val[features])
              test_preds=model.predict(test[features])

              print("VAL RMSE:",np.sqrt(mean_squared_error(val["target_share"],val_preds)))
              print("TEST RMSE:",np.sqrt(mean_squared_error(test["target_share"],test_preds)))

              test["prediction"]=test_preds
              test["alert"]=(abs(test["prediction"]-test["lag1"])>0.03).astype(int)

              plt.figure(figsize=(10,5))
              plt.plot(test["target_share"].values,label="Actual")
              plt.plot(test_preds,label="Predicted",linestyle="--")
              plt.legend()
              plt.title("Market Share Forecast")
              plt.show()

# MAIN
if __name__=="__main__":
  demand_forecasting()
  anomaly_detection()
  market_share_intelligence()
  print("\nPIPELINE COMPLETE")


